# Nuclei instance segmentation on PanNuke — method comparison

A visual, side-by-side comparison of three ways to point a segmenter at nuclei,
against the ground-truth-prompted ceiling:

| method | how it finds nuclei | family |
|---|---|---|
| **LocateAnything-3B → SAM2** | fine-tuned **grounding VLM** emits boxes | text grounding (NVIDIA) |
| **OWLv2 → SAM2** | **open-vocabulary** detector emits boxes | open-vocab detection |
| **SAM3 (text concept)** | **promptable concept segmentation**, end to end | concept segmentation |
| **oracle (GT boxes → SAM2)** | ground-truth boxes | upper bound |

**Why these four together.** A nuclei pipeline must *find*, *name*, and *segment*
each nucleus. The **oracle** fixes SAM2's segmentation ceiling, so whatever a
method loses against it is a **detection** gap, not a mask gap. LocateAnything is
purpose-tuned for grounding; OWLv2 is a generic open-vocab detector; SAM3 skips
the detector entirely. The gallery below makes the differences obvious at a glance.

> Set **`N_IMAGES`** and **`SEED`** in the config to pick how many random patches
> to show (and reshuffle). Runs on a Kaggle GPU; LocateAnything is CUDA-only.
> NVIDIA's LocateAnything license is academic / non-profit research only.


## 1 — Bootstrap
Kaggle ships a CUDA build of torch; we add only the model stacks (`transformers==4.57.1` serves LocateAnything, OWLv2, SAM2 and SAM3) and the package from GitHub. **Enable GPU + Internet** (Settings → Accelerator: GPU, Internet: On). If a previous attempt half-installed the package, **restart the kernel first** (Run → Restart & clear outputs) so the stale copy is cleared before this cell runs.

In [ ]:
import os, sys, subprocess
IN_KAGGLE = os.path.exists("/kaggle")
REPO = "git+https://github.com/marjanstoimchev/vlm-medseg.git@main"

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

if IN_KAGGLE:
    pip("transformers==4.57.1", "decord==0.6.0", "lmdb==1.7.5", "peft", "accelerate", "einops", "timm")
    # Clean rebuild of our package only: a stale/cached copy from an earlier run can be
    # missing subpackages; --force-reinstall --no-cache-dir guarantees a fresh build,
    # --no-deps leaves Kaggle's preinstalled torch/transformers untouched.
    pip("--no-cache-dir", "--force-reinstall", "--no-deps", REPO)
else:
    pip("-e", "..")

from vlm_medseg.data import get_dataset  # smoke test: fail here, not 5 cells later
print("bootstrap done")

## 2 — Configuration
**`N_IMAGES`** = how many random patches to compare · **`SEED`** = which ones (change it to reshuffle).

In [ ]:
import gc, numpy as np, torch
import matplotlib.pyplot as plt

DATASET     = "pannuke"
FOLD        = "fold1"
N_IMAGES    = 6            # random patches to compare (raise for a bigger gallery)
SEED        = 0            # change to draw a different random set
CLASS_AWARE = True         # per-class prompts -> class-coloured overlays
IOU_THRESH  = 0.5
INPUT_SHORT = 1024         # upscale 256 -> 1024 so the VLMs can see tiny nuclei
SAM3_THRESH = 0.3
OWL_THRESH  = 0.1
RUN_LA, RUN_OWLV2, RUN_SAM3 = True, True, True   # toggle methods

def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
DEVICE = "cuda" if IN_KAGGLE else "auto"
print("comparing", N_IMAGES, "random patches from", f"{DATASET}/{FOLD}", "· seed", SEED)

## 3 — Pick N random patches
Uniform-random (not stratified) so the demo reflects a typical draw — change `SEED` to resample.

In [ ]:
import random
from vlm_medseg.data import get_dataset
from vlm_medseg.viz import overlay_gt

ds = get_dataset(DATASET, fold=FOLD); spec = ds.spec
idx = random.Random(SEED).sample(range(len(ds)), N_IMAGES)
samples = [ds.decode(i) for i in idx]
print("nuclei per patch:", [s.num_instances for s in samples])

cols = min(N_IMAGES, 6)
rows = (N_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.8 * rows), squeeze=False)
for ax in axes.flat: ax.axis("off")
for ax, s in zip(axes.flat, samples):
    ax.imshow(overlay_gt(s, spec=spec))
    ax.set_title(f"{spec.group_name(s.group)} · n={s.num_instances}", fontsize=8)
fig.suptitle("Ground-truth nuclei (the targets)", y=1.02); plt.tight_layout(); plt.show()

## 4 — Run each method
SAM2 is loaded once and shared; each detector/segmenter is loaded, run over the patches, then freed (so it fits a 16 GB GPU). The oracle uses ground-truth boxes.

> **Version note.** LocateAnything pins `transformers==4.57.1`; **SAM3** (`Sam3Model`) only exists in `transformers>=5`. They cannot share one kernel, so whichever isn't importable is **skipped with a message** below — the comparison still renders for the rest. To compare **SAM3** instead of LA, set `RUN_LA=False`, add a newer `transformers` to the bootstrap, and restart.

In [ ]:
from vlm_medseg.segment.sam2 import Sam2Masker
from vlm_medseg.detect.oracle import OracleBoxDetector
from vlm_medseg.pipeline.run import run_condition, run_segmenter_condition

masker = Sam2Masker(device=DEVICE)
runs = {}

# Oracle is the segmentation ceiling -- always run it as the baseline.
runs["oracle"] = run_condition(OracleBoxDetector(class_aware=CLASS_AWARE, spec=spec),
                               masker, samples, spec=spec, iou_thresh=IOU_THRESH)

# Each optional method is isolated: a failure (e.g. a transformers version that
# lacks SAM3) skips that method with a note instead of losing the whole comparison.
if RUN_OWLV2:
    try:
        from vlm_medseg.detect.owlv2 import Owlv2Detector
        owl = Owlv2Detector(device=DEVICE, class_aware=CLASS_AWARE, spec=spec, threshold=OWL_THRESH)
        runs["owlv2"] = run_condition(owl, masker, samples, spec=spec, iou_thresh=IOU_THRESH)
        del owl; free()
    except Exception as e:
        print("owlv2 skipped:", type(e).__name__, e)

if RUN_LA:
    try:
        from vlm_medseg.detect.locate_anything import LocateAnythingDetector
        la = LocateAnythingDetector(device="cuda", class_aware=CLASS_AWARE, spec=spec, input_short_size=INPUT_SHORT)
        runs["la"] = run_condition(la, masker, samples, spec=spec, iou_thresh=IOU_THRESH)
        del la; free()
    except Exception as e:
        print("locate-anything skipped:", type(e).__name__, e)

if RUN_SAM3:
    try:
        from vlm_medseg.segment.sam3 import Sam3TextSegmenter
        sam3 = Sam3TextSegmenter(device=DEVICE, spec=spec, class_aware=CLASS_AWARE, threshold=SAM3_THRESH)
        runs["sam3"] = run_segmenter_condition(sam3, samples, spec=spec, iou_thresh=IOU_THRESH)
        del sam3; free()
    except ImportError as e:
        print("SAM3 skipped -- this transformers build has no Sam3Model:", e)
        print("  LocateAnything pins transformers==4.57.1, which predates SAM3. To include SAM3,")
        print("  set RUN_LA=False and add a newer transformers (>=5) to the bootstrap, then restart.")
    except Exception as e:
        print("SAM3 skipped:", type(e).__name__, e)

del masker; free()
print("ran:", list(runs))

## 5 — The comparison gallery
One row per patch: **H&E · ground truth · each method**. Masks are class-coloured. Read it like a contact sheet — where a method misses nuclei, leaves background, or mis-colours (e.g. SAM3 tends to collapse to one class) jumps right out.

In [ ]:
from vlm_medseg.viz import gallery
pred_by_cond = {name: [r.instances for r in out["results"]] for name, out in runs.items()}
fig = gallery(samples, pred_by_cond, spec=spec, max_rows=N_IMAGES)
fig.suptitle("H&E | GT | " + " | ".join(runs), y=1.005, fontsize=12); plt.show()

## 6 — Quantitative comparison
Binary PQ / AJI / Dice / matched-IoU (mask quality of found nuclei) + detection recall/F1 and the count ratio. The oracle row is the ceiling.

In [ ]:
import pandas as pd
def row(name, out):
    s = out["summary"]; d = s["detection_pooled"]
    r = {"method": name, "PQ": s["binary_pq"], "AJI": s["aji"], "Dice": s["dice"],
         "matchedIoU": s["matched_iou"], "det_R": d["recall"], "det_F1": d["f1"],
         "pred/gt": s["counting"]["total_pred"] / max(1, s["counting"]["total_gt"])}
    if "mpq" in s: r["mPQ"] = s["mpq"]
    return r
table = pd.DataFrame([row(k, v) for k, v in runs.items()]).set_index("method")
display(table.round(3))

from vlm_medseg.viz import plot_metric_bars
plot_metric_bars({k: v["summary"] for k, v in runs.items()}).show()

## 7 — One patch, up close
The most-populated patch, each method side by side — to inspect mask boundaries and class colours.

In [ ]:
from vlm_medseg.viz import comparison_panel
busiest = int(np.argmax([s.num_instances for s in samples]))
s = samples[busiest]
panel = {name: out["results"][busiest].instances for name, out in runs.items()}
fig = comparison_panel(s, panel, spec=spec, figsize_scale=3.4)
fig.suptitle(f"{spec.group_name(s.group)} · {s.num_instances} nuclei", y=1.03); plt.show()

## 8 — Reading the comparison

- **Oracle** = SAM2's ceiling given perfect boxes (~0.8 PQ here). Every other method's gap to it is a **detection** gap, not a segmentation one — confirmed by **matched-IoU**, which stays high (~0.8) for any nucleus a method *does* find.
- **LocateAnything** (fine-tuned grounding) vs **OWLv2** (generic open-vocab): compare their recall / count ratio — who actually localizes the nuclei.
- **SAM3** segments densely without a detector; watch its class colours — concept prompts don't separate subtypes well, so it tends to collapse to one class (the reason for the post-hoc UNI2-h classifier in the other notebooks).
- Bump **`N_IMAGES`** for a broader contact sheet, or change **`SEED`** to resample — the story should hold across draws.

*Optional:* relabel any method's masks with the trained classifier by passing
`classifier=NucleusClassifier.load("uni2h_mlp")` to `run_condition` /
`run_segmenter_condition` (attach the probe as a Kaggle dataset first).
